# Análise Preditiva Multialvo do Mercado Imobiliário em Sorocaba

## Lima Imobiliária - Projeto de Aprendizagem de Máquina de Ponta a Ponta

### Contexto de Negócio

A Lima Imobiliária possui uma carteira de imóveis com diferentes finalidades, tipos, bairros, faixas de preço e status comerciais. Além da base de imóveis, existem registros de contatos recebidos por canais digitais e uma base complementar de cadastros de clientes/proprietários.

O desafio de negócio é transformar esses dados em uma ferramenta de apoio à decisão comercial. O projeto separa duas dimensões principais:

- Resultado comercial: se o imóvel tende a ser alugado ou vendido.
- Volume esperado de leads: quantos contatos o imóvel tende a gerar.

O resultado esperado é um ranking de imóveis ativos para apoiar a priorização comercial.

### Perguntas que este projeto responde

1. Quais imóveis têm maior probabilidade de sucesso comercial?
2. Quantos contatos um imóvel tende a receber?
3. A quantidade de contatos ajuda a explicar o sucesso comercial ou pode representar vazamento de informação?
4. Quais imóveis ativos devem ser priorizados pela equipe comercial?
5. Quais características de imóvel, localização, preço e cadastro estão mais associadas a sucesso e volume de contatos?

### Bases de Dados

- `Imoveis.csv`: base central do projeto, com características do imóvel, preço, localização, finalidade, tipo e disponibilidade.
- `Contatos.csv`: histórico de contatos/leads, usado para medir demanda, volume de contatos, canais de origem, horários e recorrência.
- `Cadastros.csv`: base complementar de cadastros, usada para enriquecer os imóveis quando houver referência preenchida.

### Variáveis-Alvo do Projeto

| Tarefa | Variável-alvo | Tipo de problema | Pergunta de negócio |
|---|---|---|---|
| Resultado comercial | `sucesso_comercial` | Classificação binária | Quais imóveis tendem a virar aluguel ou venda? |
| Volume de contatos | `volume_de_contatos` | Regressão | Quantos contatos um imóvel tende a receber? |

As duas tarefas usam o mesmo `df_master`, mas com listas de features diferentes para evitar vazamento. Quando `total_contatos` é o alvo, ele não pode entrar como feature.

### Limitações Conhecidas dos Dados

- Não há data de fechamento comercial do imóvel; por isso, `total_contatos` pode conter informação posterior ao evento de venda/aluguel.
- `Cadastros.csv` tem poucas linhas com `Referência` preenchida, então entra como enriquecimento parcial.
- `Contatos.csv` contém contatos sem código de imóvel, que não podem ser ligados diretamente ao `df_master`.
- Algumas colunas são constantes ou vazias, como `Importante`, `E-mail Aberto?`, `Cidade`, `Destinatário` e `Data Abertura` em contatos.
- O alvo `Vendido` tem poucos exemplos, então venda e locação são modeladas juntas na primeira versão.

### Roteiro do Projeto

1. Definir contexto de negócio e perguntas do projeto.
2. Carregar os dados no Google Colab.
3. Inspecionar estrutura, nulos, duplicidades e colunas pouco informativas.
4. Criar uma tabela unificada com uma linha por imóvel.
5. Explorar os dados para encontrar padrões e riscos.
6. Criar os dois alvos do projeto.
7. Preparar features e pipelines de transformação.
8. Treinar e comparar modelos de classificação e regressão.
9. Avaliar os modelos com métricas adequadas.
10. Analisar erros, importância de features e risco de vazamento.
11. Gerar ranking final de imóveis ativos.
12. Persistir modelos e exportar resultados.



## 1. Instalação e Setup do Google Colab

Ambiente de execução: Google Colab. Instalar bibliotecas extras apenas se necessário.


In [ ]:
# Se necessário, descomente a linha abaixo no Colab.
# !pip install -q openpyxl joblib


## 2. Importação das bibliotecas


In [ ]:
import os
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


warnings.filterwarnings("ignore")
try:
    set_config(transform_output="pandas")
except TypeError:
    pass

RANDOM_STATE = 42


## 3. Carregamento dos dados

Arquivos necessários: `Imoveis.csv`, `Contatos.csv` e `Cadastros.csv`.


In [ ]:
ARQUIVOS_ESPERADOS = ["Imoveis.csv", "Contatos.csv", "Cadastros.csv"]

faltantes = [arquivo for arquivo in ARQUIVOS_ESPERADOS if not Path(arquivo).exists()]

if faltantes:
    try:
        from google.colab import files
        print("Envie os arquivos:", ", ".join(faltantes))
        uploaded = files.upload()
    except ModuleNotFoundError:
        print("Arquivos faltantes:", faltantes)
        print("Fora do Colab, coloque os CSVs no mesmo diretório deste notebook.")
else:
    print("Todos os arquivos esperados foram encontrados no diretório atual.")


In [ ]:
def ler_arquivo(nome_arquivo):
    caminho = Path(nome_arquivo)
    extensao = caminho.suffix.lower()

    if extensao == ".csv":
        return pd.read_csv(caminho, encoding="latin-1", sep=";")
    if extensao in [".xls", ".xlsx"]:
        return pd.read_excel(caminho)

    raise ValueError(f"Formato não suportado: {nome_arquivo}")


df_imoveis = ler_arquivo("Imoveis.csv")
df_contatos = ler_arquivo("Contatos.csv")
df_cadastros = ler_arquivo("Cadastros.csv")

for df in [df_imoveis, df_contatos, df_cadastros]:
    df.columns = df.columns.str.strip()

print("Imóveis:", df_imoveis.shape)
print("Contatos:", df_contatos.shape)
print("Cadastros:", df_cadastros.shape)


## 4. Inspeção inicial das bases

Verificação de estrutura, nulos, duplicidades e colunas com baixo valor informativo.


In [ ]:
def resumo_base(df, nome):
    resumo = pd.DataFrame({
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": df.isna().sum().values,
        "nulos_%": (df.isna().mean().values * 100).round(1),
        "unicos": df.nunique(dropna=True).values,
    })
    print(f"\n=== {nome} ===")
    print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
    display(resumo)
    display(df.head(3))

resumo_base(df_imoveis, "Imóveis")
resumo_base(df_contatos, "Contatos")
resumo_base(df_cadastros, "Cadastros")


In [ ]:
print("Distribuição de Disponibilidade:")
display(df_imoveis["Disponibilidade"].fillna("<vazio>").value_counts())

print("Tipos de contato:")
display(df_contatos["Tipo"].fillna("<vazio>").value_counts())

print("Perfis de cadastro:")
display(df_cadastros["Perfil"].fillna("<vazio>").value_counts().head(15))


## 5. Funções auxiliares de limpeza


In [ ]:
def normalizar_texto(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip()
    return texto if texto else np.nan


def converter_moeda(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip().lower()
    if not texto or "consultar" in texto:
        return np.nan
    texto = re.sub(r"[^0-9,.-]", "", texto)
    if texto.count(",") == 1:
        texto = texto.replace(".", "").replace(",", ".")
    try:
        return float(texto)
    except ValueError:
        return np.nan


def extrair_numero_padrao(texto, padrao):
    if pd.isna(texto):
        return 0
    match = re.search(padrao, str(texto).lower())
    return int(match.group(1)) if match and match.group(1) else 0


def extrair_codigo_imovel(texto):
    if pd.isna(texto):
        return pd.NA
    match = re.search(r"C[oó]d\.\s*-\s*(\d+)", str(texto), flags=re.IGNORECASE)
    return pd.to_numeric(match.group(1), errors="coerce") if match else pd.NA


def tipo_simplificado(tipo):
    if pd.isna(tipo):
        return "Outro"
    texto = str(tipo).lower()
    if "apart" in texto or "flat" in texto:
        return "Apartamento"
    if "casa" in texto or "chácara" in texto or "chacara" in texto:
        return "Casa"
    if "terreno" in texto or "lote" in texto:
        return "Terreno/Lote"
    if "comercial" in texto or "sala" in texto or "salão" in texto or "salao" in texto:
        return "Comercial"
    return "Outro"


def moda_serie(serie):
    serie = serie.dropna()
    if serie.empty:
        return np.nan
    return serie.mode().iloc[0]


def parse_data_contato(serie):
    return pd.to_datetime(
        serie.astype(str).str.replace(" às ", " ", regex=False),
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce",
    )


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


## 6. ETL dos imóveis


In [ ]:
imoveis = df_imoveis.copy()

imoveis["Referencia_num"] = pd.to_numeric(imoveis["Referência"], errors="coerce").astype("Int64")
imoveis["Valor_num"] = imoveis["Valor"].apply(converter_moeda)
imoveis["Valor_IPTU_num"] = imoveis.get("Valor IPTU", pd.Series(index=imoveis.index)).apply(converter_moeda)
imoveis["Valor_Condominio_num"] = imoveis.get("Valor Condomínio", pd.Series(index=imoveis.index)).apply(converter_moeda)
imoveis["Valor_m2_num"] = imoveis.get("Valor m²", pd.Series(index=imoveis.index)).apply(converter_moeda)

caracteristicas = imoveis.get("Características", pd.Series(index=imoveis.index))
imoveis["dormitorios"] = caracteristicas.apply(lambda x: extrair_numero_padrao(x, r"(\d+)\s*dormit"))
imoveis["suites"] = caracteristicas.apply(lambda x: extrair_numero_padrao(x, r"(\d+)\s*su[ií]te"))
imoveis["banheiros"] = caracteristicas.apply(lambda x: extrair_numero_padrao(x, r"(\d+)\s*banheiro"))
imoveis["garagens"] = caracteristicas.apply(lambda x: extrair_numero_padrao(x, r"(\d+)\s*garagen?s"))
imoveis["tipo_simples"] = imoveis["Tipo"].apply(tipo_simplificado)

for coluna in ["Finalidade", "Tipo", "Cidade", "Bairro", "Zona", "Região", "Disponibilidade"]:
    if coluna in imoveis.columns:
        imoveis[coluna] = imoveis[coluna].apply(normalizar_texto)

print("Imóveis com referência preenchida:", imoveis["Referencia_num"].notna().sum())
print("Referências duplicadas:", imoveis["Referencia_num"].duplicated().sum())
display(imoveis[["Referencia_num", "Disponibilidade", "Finalidade", "tipo_simples", "Bairro", "Valor_num", "dormitorios", "banheiros", "garagens"]].head())


## 7. ETL dos contatos

Agregação dos contatos por imóvel para medir demanda e comportamento dos leads.


In [ ]:
contatos = df_contatos.copy()
contatos["cod_imovel"] = contatos["Imóvel"].apply(extrair_codigo_imovel).astype("Int64")
contatos["data_contato"] = parse_data_contato(contatos["Data"])
contatos["hora_contato"] = contatos["data_contato"].dt.hour
contatos["dia_semana_contato"] = contatos["data_contato"].dt.day_name()

mensagem_lower = contatos["Mensagem"].fillna("").str.lower()
contatos["origem_olx"] = mensagem_lower.str.contains("olx", regex=False).astype(int)
contatos["origem_chaves_na_mao"] = (
    mensagem_lower.str.contains("chaves na mão", regex=False)
    | mensagem_lower.str.contains("chaves na mao", regex=False)
).astype(int)
contatos["origem_buskaza"] = mensagem_lower.str.contains("buskaza", regex=False).astype(int)
contatos["origem_site"] = (
    mensagem_lower.str.contains("limaimoveisnet", regex=False)
    | mensagem_lower.str.contains("site", regex=False)
).astype(int)

print("Contatos totais:", len(contatos))
print("Contatos com código de imóvel:", contatos["cod_imovel"].notna().sum())
print("Contatos sem código de imóvel:", contatos["cod_imovel"].isna().sum())
print("Imóveis únicos com contato:", contatos["cod_imovel"].nunique(dropna=True))


In [ ]:
contatos_validos = contatos.dropna(subset=["cod_imovel"]).copy()

contatos_agg = (
    contatos_validos
    .groupby("cod_imovel")
    .agg(
        total_contatos=("cod_imovel", "size"),
        qtd_nomes_contato_unicos=("Nome Contato", "nunique"),
        primeira_data_contato=("data_contato", "min"),
        ultima_data_contato=("data_contato", "max"),
        hora_modal_contato=("hora_contato", moda_serie),
        dia_semana_modal_contato=("dia_semana_contato", moda_serie),
        qtd_origem_olx=("origem_olx", "sum"),
        qtd_origem_chaves_na_mao=("origem_chaves_na_mao", "sum"),
        qtd_origem_buskaza=("origem_buskaza", "sum"),
        qtd_origem_site=("origem_site", "sum"),
    )
    .reset_index()
)

contatos_tipo = (
    pd.crosstab(contatos_validos["cod_imovel"], contatos_validos["Tipo"])
    .add_prefix("qtd_contatos_tipo_")
    .reset_index()
)

contatos_agg = contatos_agg.merge(contatos_tipo, on="cod_imovel", how="left")
contatos_agg["dias_entre_primeiro_ultimo_contato"] = (
    contatos_agg["ultima_data_contato"] - contatos_agg["primeira_data_contato"]
).dt.days

contatos_agg.head()


## 8. ETL dos cadastros

Uso de `Cadastros.csv` como enriquecimento parcial dos imóveis com referência preenchida.


In [ ]:
cadastros = df_cadastros.copy()
cadastros["Referencia_num"] = pd.to_numeric(cadastros["Referência"], errors="coerce").astype("Int64")
cadastros["Cliente_Desde_dt"] = pd.to_datetime(cadastros["Cliente Desde"], format="%d/%m/%Y", errors="coerce")

perfil_lower = cadastros["Perfil"].fillna("").str.lower()
cadastros["perfil_proprietario"] = perfil_lower.str.contains("propriet", regex=False).astype(int)
cadastros["perfil_cliente"] = perfil_lower.str.contains("cliente", regex=False).astype(int)

cadastros_validos = cadastros.dropna(subset=["Referencia_num"]).copy()

cadastros_agg = (
    cadastros_validos
    .groupby("Referencia_num")
    .agg(
        tem_cadastro_relacionado=("Referencia_num", "size"),
        qtd_cadastros_ref=("Referencia_num", "size"),
        tem_perfil_proprietario=("perfil_proprietario", "max"),
        tem_perfil_cliente=("perfil_cliente", "max"),
        captacao_principal=("Captação", moda_serie),
        cliente_desde_mais_antigo=("Cliente_Desde_dt", "min"),
    )
    .reset_index()
)

cadastros_agg["idade_cadastro_dias"] = (pd.Timestamp.today().normalize() - cadastros_agg["cliente_desde_mais_antigo"]).dt.days

print("Cadastros com referência:", cadastros_validos.shape[0])
print("Referências únicas em cadastros:", cadastros_validos["Referencia_num"].nunique())
display(cadastros_agg.head())


## 9. Criação do `df_master`


In [ ]:
df_master = imoveis.merge(
    contatos_agg,
    left_on="Referencia_num",
    right_on="cod_imovel",
    how="left",
)

df_master = df_master.merge(
    cadastros_agg,
    on="Referencia_num",
    how="left",
)

colunas_contagem = [coluna for coluna in df_master.columns if coluna.startswith("qtd_") or coluna in [
    "total_contatos",
    "qtd_nomes_contato_unicos",
    "tem_cadastro_relacionado",
    "tem_perfil_proprietario",
    "tem_perfil_cliente",
]]

for coluna in colunas_contagem:
    df_master[coluna] = df_master[coluna].fillna(0)

print("df_master:", df_master.shape)
print("Imóveis com pelo menos 1 contato:", (df_master["total_contatos"] > 0).sum())
print("Imóveis sem contato:", (df_master["total_contatos"] == 0).sum())
print("Imóveis com cadastro relacionado:", (df_master["tem_cadastro_relacionado"] > 0).sum())

display(df_master[["Referencia_num", "Disponibilidade", "Finalidade", "tipo_simples", "Bairro", "Valor_num", "total_contatos", "tem_cadastro_relacionado"]].head())


## 10. Criação dos dois alvos


In [ ]:
status = df_master["Disponibilidade"].fillna("").str.strip().str.lower()

positivos_sucesso = ["alugado", "vendido"]
negativos_sucesso = ["desistência", "desistencia", "outros"]

df_master["sucesso_comercial"] = np.nan
df_master.loc[status.isin(positivos_sucesso), "sucesso_comercial"] = 1
df_master.loc[status.isin(negativos_sucesso), "sucesso_comercial"] = 0
df_master["sucesso_comercial"] = df_master["sucesso_comercial"].astype("float")

df_master["volume_de_contatos"] = df_master["total_contatos"].astype(float)
df_master["log_volume_de_contatos"] = np.log1p(df_master["volume_de_contatos"])

print("Sucesso comercial:")
display(df_master["sucesso_comercial"].value_counts(dropna=False))


print("Volume de contatos:")
display(df_master["volume_de_contatos"].describe())


## 11. EDA orientada aos alvos


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df_master["Disponibilidade"].fillna("<vazio>").value_counts().plot(kind="bar", ax=axes[0], color="#2f6f73")
axes[0].set_title("Disponibilidade")
axes[0].set_ylabel("Imóveis")

np.log1p(df_master["total_contatos"]).hist(bins=30, ax=axes[1], color="#bf7d30")
axes[1].set_title("Distribuição de log1p(total_contatos)")
axes[1].set_xlabel("log1p(contatos)")

media_contatos_tipo = df_master.groupby("tipo_simples")["total_contatos"].mean().sort_values()
media_contatos_tipo.plot(kind="barh", ax=axes[2], color="#425c8a")
axes[2].set_title("Média de contatos por tipo")
axes[2].set_xlabel("Média de contatos")

plt.tight_layout()
plt.show()


In [ ]:
top_bairros = (
    df_master.groupby("Bairro", dropna=True)
    .agg(total_contatos=("total_contatos", "sum"), qtd_imoveis=("Referencia_num", "count"), media_contatos=("total_contatos", "mean"))
    .sort_values("total_contatos", ascending=False)
    .head(20)
)

display(top_bairros)

ax = top_bairros["total_contatos"].sort_values().plot(kind="barh", figsize=(10, 7), color="#2f6f73")
ax.set_title("Top 20 bairros por volume total de contatos")
ax.set_xlabel("Total de contatos")
plt.show()


In [ ]:
df_master["faixa_contatos"] = pd.cut(
    df_master["total_contatos"],
    bins=[-1, 0, 5, 10, 30, np.inf],
    labels=["0", "1-5", "6-10", "11-30", "31+"],
)

analise_sucesso_volume = (
    df_master.dropna(subset=["sucesso_comercial"])
    .groupby("faixa_contatos", observed=False)
    .agg(
        qtd_imoveis=("Referencia_num", "count"),
        taxa_sucesso=("sucesso_comercial", "mean"),
        media_contatos=("total_contatos", "mean"),
    )
)

display(analise_sucesso_volume)


## 12. Features e pipelines

As features são separadas por modelo para evitar vazamento de informação.


In [ ]:
FEATURES_BASE_NUM = [
    "Valor_num",
    "Valor_IPTU_num",
    "Valor_Condominio_num",
    "Valor_m2_num",
    "dormitorios",
    "suites",
    "banheiros",
    "garagens",
    "tem_cadastro_relacionado",
    "qtd_cadastros_ref",
    "tem_perfil_proprietario",
    "tem_perfil_cliente",
    "idade_cadastro_dias",
]

FEATURES_BASE_CAT = [
    "Finalidade",
    "tipo_simples",
    "Bairro",
    "Cidade",
    "Zona",
    "Região",
    "captacao_principal",
]

FEATURES_CONTATOS_NUM = [
    "total_contatos",
    "qtd_nomes_contato_unicos",
    "dias_entre_primeiro_ultimo_contato",
    "qtd_origem_olx",
    "qtd_origem_chaves_na_mao",
    "qtd_origem_buskaza",
    "qtd_origem_site",
]

FEATURES_CONTATOS_CAT = [
    "hora_modal_contato",
    "dia_semana_modal_contato",
]

# Mantém apenas colunas existentes e com pelo menos um valor não nulo.
def existentes(colunas):
    return [
        coluna for coluna in colunas
        if coluna in df_master.columns and df_master[coluna].notna().any()
    ]

FEATURES_BASE_NUM = existentes(FEATURES_BASE_NUM)
FEATURES_BASE_CAT = existentes(FEATURES_BASE_CAT)
FEATURES_CONTATOS_NUM = existentes(FEATURES_CONTATOS_NUM)
FEATURES_CONTATOS_CAT = existentes(FEATURES_CONTATOS_CAT)

FEATURES_SUCESSO_COM_CONTATOS = FEATURES_BASE_NUM + FEATURES_BASE_CAT + FEATURES_CONTATOS_NUM + FEATURES_CONTATOS_CAT
FEATURES_SUCESSO_SEM_CONTATOS = FEATURES_BASE_NUM + FEATURES_BASE_CAT
FEATURES_VOLUME = FEATURES_BASE_NUM + FEATURES_BASE_CAT

print("Features sucesso com contatos:", len(FEATURES_SUCESSO_COM_CONTATOS))
print("Features sucesso sem contatos:", len(FEATURES_SUCESSO_SEM_CONTATOS))
print("Features volume:", len(FEATURES_VOLUME))


In [ ]:
def criar_preprocessador(features, base_df=None):
    if base_df is None:
        base_df = df_master
    numericas = [coluna for coluna in features if pd.api.types.is_numeric_dtype(base_df[coluna])]
    categoricas = [coluna for coluna in features if coluna not in numericas]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numericas),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_one_hot_encoder()),
            ]), categoricas),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return preprocessor


def avaliar_classificacao(nome, modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]

    metricas = {
        "modelo": nome,
        "auc_roc": roc_auc_score(y_test, y_prob),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }

    print(f"\n=== {nome} ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("AUC-ROC:", round(metricas["auc_roc"], 4))

    return modelo, metricas, y_pred, y_prob


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


def avaliar_regressao(nome, modelo, X_train, X_test, y_train_log, y_test_log):
    modelo.fit(X_train, y_train_log)
    pred_log = modelo.predict(X_test)

    y_test_original = np.expm1(y_test_log)
    pred_original = np.clip(np.expm1(pred_log), 0, None)

    metricas = {
        "modelo": nome,
        "mae": mean_absolute_error(y_test_original, pred_original),
        "rmse": rmse(y_test_original, pred_original),
        "r2": r2_score(y_test_original, pred_original),
    }

    print(f"\n=== {nome} ===")
    print("MAE:", round(metricas["mae"], 3))
    print("RMSE:", round(metricas["rmse"], 3))
    print("R2:", round(metricas["r2"], 3))

    return modelo, metricas, pred_original


## 13. Modelo 1 - Sucesso comercial

Classificação binária para prever imóveis com desfecho `Alugado` ou `Vendido`.

Serão comparadas duas versões:

- Com dados de contatos.
- Sem dados de contatos.


In [ ]:
df_sucesso = df_master.dropna(subset=["sucesso_comercial"]).copy()
df_sucesso["sucesso_comercial"] = df_sucesso["sucesso_comercial"].astype(int)

print("Dataset sucesso comercial:", df_sucesso.shape)
display(df_sucesso["sucesso_comercial"].value_counts())


In [ ]:
def treinar_modelos_classificacao(df_modelo, target, features, titulo):
    X = df_modelo[features].copy()
    y = df_modelo[target].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    preprocessor = criar_preprocessador(features, base_df=df_modelo)

    modelos = {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }

    resultados = []
    modelos_treinados = {}

    print(f"\n### {titulo}")
    for nome, estimador in modelos.items():
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", estimador),
        ])
        modelo_treinado, metricas, y_pred, y_prob = avaliar_classificacao(nome, pipeline, X_train, X_test, y_train, y_test)
        resultados.append(metricas)
        modelos_treinados[nome] = modelo_treinado

    resultados_df = pd.DataFrame(resultados).sort_values("auc_roc", ascending=False)
    display(resultados_df)

    return modelos_treinados, resultados_df, (X_train, X_test, y_train, y_test)

modelos_sucesso_com, resultados_sucesso_com, dados_sucesso_com = treinar_modelos_classificacao(
    df_sucesso,
    "sucesso_comercial",
    FEATURES_SUCESSO_COM_CONTATOS,
    "Sucesso comercial com contatos",
)

modelos_sucesso_sem, resultados_sucesso_sem, dados_sucesso_sem = treinar_modelos_classificacao(
    df_sucesso,
    "sucesso_comercial",
    FEATURES_SUCESSO_SEM_CONTATOS,
    "Sucesso comercial sem contatos",
)


In [ ]:
comparacao_vazamento = pd.concat([
    resultados_sucesso_com.assign(versao="com_contatos"),
    resultados_sucesso_sem.assign(versao="sem_contatos"),
], ignore_index=True).sort_values(["auc_roc", "f1"], ascending=False)

display(comparacao_vazamento)

melhor_sucesso_linha = comparacao_vazamento.iloc[0]
if melhor_sucesso_linha["versao"] == "com_contatos":
    modelo_sucesso_final = modelos_sucesso_com[melhor_sucesso_linha["modelo"]]
    features_sucesso_final = FEATURES_SUCESSO_COM_CONTATOS
    dados_sucesso_final = dados_sucesso_com
else:
    modelo_sucesso_final = modelos_sucesso_sem[melhor_sucesso_linha["modelo"]]
    features_sucesso_final = FEATURES_SUCESSO_SEM_CONTATOS
    dados_sucesso_final = dados_sucesso_sem

print("Modelo final de sucesso comercial:", melhor_sucesso_linha.to_dict())


## 14. Modelo 2 - Volume de contatos

Regressão para prever `total_contatos`.

O alvo de treino usa `log1p(total_contatos)` para reduzir o impacto de valores extremos.


In [ ]:
df_demanda_regressao = df_master.copy()

X = df_demanda_regressao[FEATURES_VOLUME].copy()
y_log = df_demanda_regressao["log_volume_de_contatos"].astype(float)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_log, test_size=0.2, random_state=RANDOM_STATE
)

preprocessor_reg = criar_preprocessador(FEATURES_VOLUME, base_df=df_demanda_regressao)

modelos_regressao = {
    "DummyMedian": DummyRegressor(strategy="median"),
    "Ridge": Ridge(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados_regressao = []
modelos_volume = {}

for nome, estimador in modelos_regressao.items():
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor_reg)),
        ("model", estimador),
    ])
    modelo_treinado, metricas, pred_original = avaliar_regressao(nome, pipeline, X_train_reg, X_test_reg, y_train_reg, y_test_reg)
    resultados_regressao.append(metricas)
    modelos_volume[nome] = modelo_treinado

resultados_regressao = pd.DataFrame(resultados_regressao).sort_values("mae")
display(resultados_regressao)

# O DummyMedian pode vencer no MAE porque mais da metade dos imóveis tem zero contatos.
# Para scoring/ranking, usamos o melhor modelo não trivial, mantendo o Dummy apenas como baseline.
resultados_regressao_validos = resultados_regressao[resultados_regressao["modelo"] != "DummyMedian"].copy()
melhor_volume_nome = resultados_regressao_validos.iloc[0]["modelo"]
modelo_volume_final = modelos_volume[melhor_volume_nome]

print("Baseline de regressão:", resultados_regressao[resultados_regressao["modelo"] == "DummyMedian"].iloc[0].to_dict())
print("Modelo final de volume de contatos:", melhor_volume_nome)


## 15. Análise dos melhores modelos


In [ ]:
def obter_feature_names(modelo):
    preprocessor = modelo.named_steps["preprocessor"]
    try:
        return preprocessor.get_feature_names_out()
    except Exception:
        return None


def mostrar_importancias_ou_coeficientes(modelo, titulo, top_n=15):
    estimator = modelo.named_steps["model"]
    nomes = obter_feature_names(modelo)
    if nomes is None:
        print(f"Não foi possível obter nomes de features para {titulo}.")
        return

    if hasattr(estimator, "coef_"):
        valores = estimator.coef_[0]
        df_imp = pd.DataFrame({"feature": nomes, "valor": valores})
        print(f"\nTop positivos - {titulo}")
        display(df_imp.sort_values("valor", ascending=False).head(top_n))
        print(f"\nTop negativos - {titulo}")
        display(df_imp.sort_values("valor", ascending=True).head(top_n))
    elif hasattr(estimator, "feature_importances_"):
        valores = estimator.feature_importances_
        df_imp = pd.DataFrame({"feature": nomes, "valor": valores})
        print(f"\nImportâncias - {titulo}")
        display(df_imp.sort_values("valor", ascending=False).head(top_n))
    else:
        print(f"Modelo {titulo} não possui coeficientes/importâncias diretas.")

mostrar_importancias_ou_coeficientes(modelo_sucesso_final, "Sucesso comercial")
mostrar_importancias_ou_coeficientes(modelo_volume_final, "Volume de contatos")


## 16. Scoring final dos imóveis ativos

Ranking dos imóveis ativos combinando:

- Probabilidade de sucesso comercial.
- Volume previsto de contatos.


In [ ]:
disponibilidade_lower = df_master["Disponibilidade"].fillna("").str.lower()
status_ativo = df_master["Disponibilidade"].isna() | disponibilidade_lower.isin(["disponível", "disponivel", ""])
df_scoring = df_master[status_ativo].copy()

if df_scoring.empty:
    print("Não há imóveis ativos para scoring conforme a regra definida.")
else:
    df_scoring["prob_sucesso_comercial"] = modelo_sucesso_final.predict_proba(df_scoring[features_sucesso_final])[:, 1]
    df_scoring["volume_contatos_previsto"] = np.clip(np.expm1(modelo_volume_final.predict(df_scoring[FEATURES_VOLUME])), 0, None)

    max_volume_previsto = df_scoring["volume_contatos_previsto"].max()
    if max_volume_previsto > 0:
        df_scoring["score_volume_contatos"] = df_scoring["volume_contatos_previsto"] / max_volume_previsto
    else:
        df_scoring["score_volume_contatos"] = 0

    df_scoring["score_priorizacao"] = (
        0.7 * df_scoring["prob_sucesso_comercial"]
        + 0.3 * df_scoring["score_volume_contatos"]
    )

    colunas_ranking = [
        "Referencia_num",
        "Tipo",
        "Finalidade",
        "Bairro",
        "Valor_num",
        "total_contatos",
        "prob_sucesso_comercial",
        "volume_contatos_previsto",
        "score_volume_contatos",
        "score_priorizacao",
    ]

    ranking_imoveis_ativos = df_scoring[colunas_ranking].sort_values("score_priorizacao", ascending=False)
    display(ranking_imoveis_ativos.head(30))

    print("Resumo do volume previsto:")
    display(ranking_imoveis_ativos["volume_contatos_previsto"].describe())


## 17. Persistência dos modelos e exportação dos resultados


In [ ]:
joblib.dump(modelo_sucesso_final, "modelo_sucesso_comercial.pkl")
joblib.dump(modelo_volume_final, "modelo_volume_contatos.pkl")

resumo_metricas = comparacao_vazamento.assign(tarefa="sucesso_comercial")

with pd.ExcelWriter("ranking_imoveis_ativos_multialvo.xlsx", engine="openpyxl") as writer:
    if "ranking_imoveis_ativos" in globals():
        ranking_imoveis_ativos.to_excel(writer, sheet_name="ranking_ativos", index=False)
    resumo_metricas.to_excel(writer, sheet_name="metricas_classificacao", index=False)
    resultados_regressao.to_excel(writer, sheet_name="metricas_regressao", index=False)

print("Arquivos gerados:")
print("- modelo_sucesso_comercial.pkl")
print("- modelo_volume_contatos.pkl")
print("- ranking_imoveis_ativos_multialvo.xlsx")


In [ ]:
try:
    from google.colab import files
    for arquivo in [
        "modelo_sucesso_comercial.pkl",
        "modelo_volume_contatos.pkl",
        "ranking_imoveis_ativos_multialvo.xlsx",
    ]:
        files.download(arquivo)
except ModuleNotFoundError:
    print("Downloads automáticos disponíveis apenas no Google Colab.")


## 18. Conclusão executiva

Pontos para preencher após a execução:

1. Melhor modelo para sucesso comercial.
2. Diferença entre modelo com contatos e sem contatos.
3. Principais fatores associados a sucesso comercial.
4. Utilidade do volume previsto de contatos no ranking.
5. Imóveis ativos recomendados para priorização comercial.

### Limitações

- Não há data de fechamento comercial do imóvel.
- `total_contatos` pode conter informação posterior ao desfecho comercial.
- `Cadastros.csv` tem poucas referências preenchidas.
- Contatos sem código de imóvel não entram no `df_master`.
- A base é pequena para separar venda e locação em modelos independentes.

### Próximos passos

- Incluir data de fechamento ou baixa do imóvel, se disponível.
- Separar modelos de venda e locação quando houver mais dados.
- Validar o ranking com a equipe comercial.
- Acompanhar a performance com novos imóveis e contatos.
